In [ ]:
repository_filter: list[str] = []

In [ ]:
from moderne_visualizations_misc.reusable.data_loader import read_data_table
from code_data_science import data_table as dt, tree_data_grid
from code_data_science import unique_dictionaries as ud
from moderne_visualizations_misc.reusable import quality_utils as qu
import warnings

warnings.simplefilter("ignore")

# df = read_data_table("../samples/method_quality_metrics.csv")
df = read_data_table("../samples/v2/io.moderne.prethink.table.MethodQualityMetrics.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)
df = qu.add_class_short(df)

In [ ]:
if len(df) == 0:
    import plotly.graph_objects as go

    fig = qu.empty_figure()
    fig.show()
else:
    # Build tree: repo -> package -> class -> method
    df["package"] = df["className"].apply(qu.extract_package)

    tree = ud.UniqueDictionaries()
    for _, row in df.iterrows():
        repo = row["repoShort"]
        package = row["package"] if row["package"] else "(default)"
        class_short = row["classShort"]
        method = row["methodName"]
        path = f"{repo}/{package}/{class_short}/{method}"

        tree.add(
            {
                "path": path,
                "methodName": method,
                "cyclomaticComplexity": str(row["cyclomaticComplexity"]),
                "cognitiveComplexity": str(row["cognitiveComplexity"]),
                "lineCount": str(row["lineCount"]),
                "parameterCount": str(row["parameterCount"]),
                "abcScore": str(row["abcScore"]),
            }
        )

    tree_data_grid.display(
        tree.to_list(),
        "Code Quality",
        [
            {
                "field": "methodName",
                "headerName": "Method",
                "minWidth": 200,
            },
            {
                "field": "cyclomaticComplexity",
                "headerName": "Cyclomatic",
                "minWidth": 110,
            },
            {
                "field": "cognitiveComplexity",
                "headerName": "Cognitive",
                "minWidth": 110,
            },
            {
                "field": "lineCount",
                "headerName": "Lines",
                "minWidth": 80,
            },
            {
                "field": "parameterCount",
                "headerName": "Params",
                "minWidth": 80,
            },
            {
                "field": "abcScore",
                "headerName": "ABC Score",
                "minWidth": 100,
            },
        ],
    )